# Prediction Intervals
One of the primary uses of regression is to make predictions for a new individual who was not part of our original sample but is similar to the sampled individuals. In the language of the model, we want to estimate $y$ for a new value of $x$.

Our estimate is the height of the true line at $x$. Of course, we don't know the true line. What we have as a substitute is the regression line through our sample of points.

The **fitted value** at a given value of $x$ is the regression estimate of $y$ based on that value of $x$. In other words, the fitted value at a given value of $x$ is the height of the regression line at that $x$.

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
import holoviews as hv
from datascience import make_array, percentile
path_data = '../../../assets/data/'

def hist_column(values, xlabel, bins=None):
    """POLARS_CHANGE: hvplot.hist draws a density, matching Table.hist and visualizations.ipynb."""
    frame = pl.DataFrame([pl.Series(xlabel, values)])
    if bins is None:
        bins = 10
    return frame.hvplot.hist(xlabel, bins=bins, normed=True, xlabel=xlabel)

In [2]:
baby = pl.read_csv(path_data + 'baby.csv')

In [3]:
def standard_units(any_numbers):
    "Convert any Series of numbers to standard units."
    return (any_numbers - any_numbers.mean())/any_numbers.std(ddof=0)  

def correlation(t, x, y):
    return (standard_units(t.get_column(x))*standard_units(t.get_column(y))).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() - a * table.get_column(x).mean()

def fit(table, x, y):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

def residual(table, x, y):
    return table.get_column(y) - fit(table, x, y)

def scatter_fit(table, x, y):
    """POLARS_CHANGE: hvplot.scatter with the regression line overlaid in yellow with a thin black outline. The chart is returned so a cell can show it or draw on it."""
    points = table.hvplot.scatter(x=x, y=y, color='#002fa7')
    fitted = pl.DataFrame([table.get_column(x), fit(table, x, y).alias('Fitted value')]).sort(x)
    line = (
        fitted.hvplot.line(x=x, y='Fitted value', color='black', line_width=3)  # dark outline for contrast
        * fitted.hvplot.line(x=x, y='Fitted value', color='#FFFF00', line_width=2)
    )
    return (points * line).opts(xlabel=x, ylabel=y)

Suppose we try to predict a baby's birth weight based on the number of gestational days. As we saw in the previous section, the data fit the regression model fairly well and a 95% confidence interval for the slope of the true line doesn't contain 0. So it seems reasonable to carry out our prediction.

The figure below shows where the prediction lies on the regression line. The red line is at $x = 300$.

In [4]:
chart = scatter_fit(baby, 'Gestational Days', 'Birth Weight')
s = slope(baby, 'Gestational Days', 'Birth Weight')
i = intercept(baby, 'Gestational Days', 'Birth Weight')
fit_300 = s*300 + i
point = hv.Scatter(([300], [fit_300])).opts(color='red', size=6)
height = hv.Curve(([300,300], [0, fit_300])).opts(color='red', line_width=2)
(chart * point * height).opts(ylim=(0, 200))

Scatterplot with 'Gestational Days' on the x-axis and 'Birth Weight' on the y-axis. The data points are shown in blue and generally have x values greater than x=225 and a weak to moderate positive correlation. Some outliers exist with low x values, around 150, 180, and just above 200. Each of these outlier data points have y values below 120. A yellow line is shown with a positive slope through the majority of the datapoints. A vertical red line is drawn from y=0 to about y=125, until it reaches the yellow line at x=300.

The height of the point where the red line hits the regression line is the fitted value at 300 gestational days. 

The function `fitted_value` computes this height. Like the functions `correlation`, `slope`, and `intercept`, its arguments include the name of the table and the labels of the $x$ and $y$ columns. But it also requires a fourth argument, which is the value of $x$ at which the estimate will be made.

In [5]:
def fitted_value(table, x, y, given_x):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * given_x  + b

The fitted value at 300 gestational days is about 129.2 ounces. In other words, for a pregnancy that has a duration of 300 gestational days, our estimate for the baby's weight is about 129.2 ounces.

In [6]:
fit_300 = fitted_value(baby, 'Gestational Days', 'Birth Weight', 300)
fit_300

129.2129241703143

## The Variability of the Prediction

We have developed a method making one prediction of a new baby's birth weight based on the number of gestational days, using the data in our sample. But as data scientists, we know that the sample might have been different. Had the sample been different, the regression line would have been different too, and so would our prediction. To see how good our prediction is, we must get a sense of how variable the prediction can be.

To do this, we must generate new samples. We can do that by bootstrapping the scatter plot as in the previous section. We will then fit the regression line to the scatter plot in each replication, and make a prediction based on each line. The figure below shows 10 such lines, and the corresponding predicted birth weight at 300 gestational days.

In [7]:
x = 300

lines = pl.DataFrame(schema={'slope': pl.Float64, 'intercept': pl.Float64})
for i in np.arange(10):
    rep = baby.sample(n=baby.height, with_replacement=True)
    a = slope(rep, 'Gestational Days', 'Birth Weight')
    b = intercept(rep, 'Gestational Days', 'Birth Weight')
    lines = pl.concat([lines, pl.DataFrame([pl.Series('slope', make_array(a)), pl.Series('intercept', make_array(b))])])

lines = lines.with_columns((pl.col('slope')*x + pl.col('intercept')).alias('prediction at x='+str(x)))

xlims = make_array(291, 309)
left = lines.get_column('slope')*xlims[0] + lines.get_column('intercept')
right = lines.get_column('slope')*xlims[1] + lines.get_column('intercept')
fit_x = lines.get_column('slope')*x + lines.get_column('intercept')

graphs = []
for i in np.arange(10):
    graphs.append(hv.Curve((xlims, make_array(left.item(i), right.item(i)))).opts(line_width=1))
    graphs.append(hv.Scatter(([x], [fit_x.item(i)])).opts(size=6))
hv.Overlay(graphs)

A plot with x ranging from 292.5 to 307.5 nd y from 124 to 136. Ten different colored lines are drawn at with an intersecting dot at x=300. The lines intersect with x=300 at y values between 126 and 131.

The predictions vary from one line to the next. The table below shows the slope and intercept of each of the 10 lines, along with the prediction. 

In [8]:
lines

slope,intercept,prediction at x=300
f64,f64,f64
0.364702,17.147982,126.558443
0.413712,3.455863,127.569541
0.475643,-13.657354,129.035544
0.427386,0.272359,128.488052
0.457739,-7.535066,129.78664
0.42206,2.054277,128.672134
0.446347,-5.454279,128.449888
0.415236,3.470082,128.040753
0.411624,4.829009,128.316064


## Bootstrap Prediction Interval

If we increase the number of repetitions of the resampling process, we can generate an empirical histogram of the predictions. This will allow us to create an interval of predictions, using the same percentile method that we used create a bootstrap confidence interval for the slope.

Let us define a function called ``bootstrap_prediction`` to do this. The function takes five arguments:
- the name of the table
- the column labels of the predictor and response variables, in that order
- the value of $x$ at which to make the prediction
- the desired number of bootstrap repetitions

In each repetition, the function bootstraps the original scatter plot and finds the predicted value of $y$ based on the specified value of $x$. Specifically, it calls the function `fitted_value` that we defined earlier in this section to find the fitted value at the specified $x$.

Finally, it draws the empirical histogram of all the predicted values, and prints the interval consisting of the "middle 95%" of the predicted values. It also prints the predicted value based on the regression line through the original scatter plot.

In [9]:
# Bootstrap prediction of variable y at new_x
# Data contained in table; prediction by regression of y based on x
# repetitions = number of bootstrap replications of the original scatter plot

def bootstrap_prediction(table, x, y, new_x, repetitions):
    
    # For each repetition:
    # Bootstrap the scatter; 
    # get the regression prediction at new_x; 
    # augment the predictions list
    predictions = make_array()
    for i in np.arange(repetitions):
        bootstrap_sample = table.sample(n=table.height, with_replacement=True)
        bootstrap_prediction = fitted_value(bootstrap_sample, x, y, new_x)
        predictions = np.append(predictions, bootstrap_prediction)
        
    # Find the ends of the approximate 95% prediction interval
    left = percentile(2.5, predictions)
    right = percentile(97.5, predictions)
    
    # Prediction based on original sample
    original = fitted_value(table, x, y, new_x)
    
    # Display results
    chart = hist_column(predictions, 'Prediction', bins=20).opts(xlabel='predictions at x='+str(new_x))
    interval = (
        hv.Curve((make_array(left, right), make_array(0, 0))).opts(color='black', line_width=9)  # dark outline for contrast
        * hv.Curve((make_array(left, right), make_array(0, 0))).opts(color='#FFFF00', line_width=8)
    )
    print('Height of regression line at x='+str(new_x)+':', original)
    print('Approximate 95%-confidence interval:')
    print(left, right)
    return chart * interval

In [10]:
bootstrap_prediction(baby, 'Gestational Days', 'Birth Weight', 300, 5000)

Height of regression line at x=300: 129.2129241703143
Approximate 95%-confidence interval:
127.342782332 131.404791105


Histogram with 'predictions at x=300' on the x-axis and 'Percent per unit' on the y-axis. The histogram is roughly bell shaped and symmetric about x~129, ranging from x=126 to x=132. A yellow bar is drawn at the bottom of the histogram from about x=127 to about x=131.

The figure above shows a bootstrap empirical histogram of the predicted birth weight of a baby at 300 gestational days, based on 5,000 repetitions of the bootstrap process. The empirical distribution is roughly normal. 

An approximate 95% prediction interval of scores has been constructed by taking the "middle 95%" of the predictions, that is, the interval from the 2.5th percentile to the 97.5th percentile of the predictions. The interval ranges from about 127 to about 131. The prediction based on the original sample was about 129, which is close to the center of the interval.

## The Effect of Changing the Value of the Predictor

The figure below shows the histogram of 5,000 bootstrap predictions at 285 gestational days. The prediction based on the original sample is about 122 ounces, and the interval ranges from about 121 ounces to about 123 ounces. 

In [11]:
bootstrap_prediction(baby, 'Gestational Days', 'Birth Weight', 285, 5000)

Height of regression line at x=285: 122.21457101607608
Approximate 95%-confidence interval:
121.169331843 123.28553839


Histogram with 'predictions at x=285' on the x-axis and 'Percent per unit' on the y-axis. The histogram is roughly bell shaped and symmetric about x~122.25, ranging from x=121 to x=124. A yellow bar is drawn at the bottom of the histogram from about x=121 to about x=123.

Notice that this interval is narrower than the prediction interval at 300 gestational days. Let us investigate the reason for this.

The mean number of gestational days is about 279 days: 

In [12]:
baby.get_column('Gestational Days').mean()

279.1013628620102

So 285 is nearer to the center of the distribution than 300 is. Typically, the regression lines based on the bootstrap samples are closer to each other near the center of the distribution of the predictor variable. Therefore all of the predicted values are closer together as well. This explains the narrower width of the prediction interval. 

You can see this in the figure below, which shows predictions at $x = 285$ and $x = 300$ for each of ten bootstrap replications. Typically, the lines are farther apart at $x = 300$ than at $x = 285$, and therefore the predictions at $x = 300$ are more variable.

In [13]:
x1 = 300
x2 = 285

lines = pl.DataFrame(schema={'slope': pl.Float64, 'intercept': pl.Float64})
for i in np.arange(10):
    rep = baby.sample(n=baby.height, with_replacement=True)
    a = slope(rep, 'Gestational Days', 'Birth Weight')
    b = intercept(rep, 'Gestational Days', 'Birth Weight')
    lines = pl.concat([lines, pl.DataFrame([pl.Series('slope', make_array(a)), pl.Series('intercept', make_array(b))])])

xlims = make_array(260, 310)
left = lines.get_column('slope')*xlims[0] + lines.get_column('intercept')
right = lines.get_column('slope')*xlims[1] + lines.get_column('intercept')
fit_x1 = lines.get_column('slope')*x1 + lines.get_column('intercept')
fit_x2 = lines.get_column('slope')*x2 + lines.get_column('intercept')

graphs = []
for i in np.arange(10):
    graphs.append(hv.Curve((xlims, make_array(left.item(i), right.item(i)))).opts(line_width=1))
    graphs.append(hv.Scatter(([x1], [fit_x1.item(i)])).opts(size=6))
    graphs.append(hv.Scatter(([x2], [fit_x2.item(i)])).opts(size=6))
hv.Overlay(graphs).opts(xlim=(xlims[0], xlims[1]))

10 lines are drawn in different colors with data points highlighted on x=285 and x=300.

### Words of caution ###

All of the predictions and tests that we have performed in this chapter assume that the regression model holds. Specifically, the methods assume that the scatter plot resembles points generated by starting with points that are on a straight line and then pushing them off the line by adding random normal noise.

If the scatter plot does not look like that, then perhaps the model does not hold for the data. If the model does not hold, then calculations that assume the model to be true are not valid.

Therefore, we must first decide whether the regression model holds for our data, before we start making predictions based on the model or testing hypotheses about parameters of the model. A simple way is to do what we did in this section, which is to draw the scatter diagram of the two variables and see whether it looks roughly linear and evenly spread out around a line. We should also run the diagnostics we developed in the previous section using the residual plot.